In [15]:
!pip install scikeras

# Uninstall the current scikit-learn version
!pip uninstall scikit-learn -y

# Install a compatible version of scikit-learn (e.g., 1.4.2)
!pip install scikit-learn==1.4.2

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
cuml-cu12 26.2.0 requires scikit-learn>=1.5, but you have scikit-learn 1.4.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler


In [2]:
# Read training data
train_df = pd.read_csv('train.csv')
train_df

,match_id,color,rank,map_code,duration,car_name,possession_time,time_in_side,shots,shots_against,...,percent_defensive_half,percent_offensive_half,percent_behind_ball,percent_infront_ball,percent_most_back,percent_most_forward,percent_closest_to_ball,percent_farthest_from_ball,demos_inflicted,demos_taken
0,0,blue,silver,neotokyo_standard_p,163,Breakout,54.95,58.96,2,4,...,62.736664,37.263340,66.842890,33.157120,95.534080,95.534080,95.534080,95.534080,1,0
1,0,orange,silver,neotokyo_standard_p,163,Octane,27.51,80.68,4,2,...,63.576576,36.423428,77.846670,22.153326,98.304170,98.304170,98.304170,98.304170,0,1
2,1,blue,gold,stadium_foggy_p,460,Octane,132.98,244.72,7,10,...,75.199120,24.800879,73.992320,26.007679,96.942440,96.942440,96.942440,96.942440,0,0
3,1,orange,gold,stadium_foggy_p,460,Octane,102.55,148.84,10,7,...,55.832005,44.167990,77.792800,22.207201,96.918510,96.918510,96.918510,96.918510,0,0
4,2,blue,silver,neotokyo_standard_p,94,Octane,25.24,33.70,0,3,...,76.376495,23.623503,68.454840,31.545156,93.636470,93.636470,93.636470,93.636470,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60237,30118,orange,silver,chn_stadium_p,390,Ronin GXT,94.20,193.21,8,6,...,64.023270,35.976734,72.403404,27.596594,98.048460,98.048460,98.048460,98.048460,0,1
60238,30119,blue,silver,trainstation_p,372,Fennec,145.58,123.44,8,3,...,56.082024,43.917976,78.380820,21.619183,98.296730,98.296730,98.296730,98.296730,0,2
60239,30119,orange,silver,trainstation_p,372,Octane,119.14,213.29,3,8,...,72.444016,27.555986,70.517235,29.482769,100.418655,100.418655,100.418655,100.418655,2,0
60240,30120,blue,platinum,wasteland_night_s_p,425,Octane,131.41,184.57,7,7,...,58.544390,41.455605,73.541850,26.458149,97.486570,97.486570,97.486570,97.486570,1,0


In [3]:
# Create dataset
match_df = train_df.groupby(['match_id', 'rank']).sum().reset_index()
match_df

,match_id,rank,color,map_code,duration,car_name,possession_time,time_in_side,shots,shots_against,...,percent_defensive_half,percent_offensive_half,percent_behind_ball,percent_infront_ball,percent_most_back,percent_most_forward,percent_closest_to_ball,percent_farthest_from_ball,demos_inflicted,demos_taken
0,0,silver,blueorange,neotokyo_standard_pneotokyo_standard_p,326,BreakoutOctane,82.46,139.64,6,6,...,126.313240,73.686768,144.689560,55.310446,193.838250,193.838250,193.838250,193.838250,1,1
1,1,gold,blueorange,stadium_foggy_pstadium_foggy_p,920,OctaneOctane,235.53,393.56,17,17,...,131.031125,68.968869,151.785120,48.214880,193.860950,193.860950,193.860950,193.860950,0,0
2,2,silver,blueorange,neotokyo_standard_pneotokyo_standard_p,188,OctaneOctane,43.25,81.11,3,3,...,131.124745,68.875258,144.769240,55.230766,187.390130,187.390130,187.390130,187.390130,0,0
3,3,platinum,blueorange,stadium_day_pstadium_day_p,686,FennecOctane,229.37,301.88,12,12,...,125.121784,74.878216,152.124215,47.875778,195.873688,195.873688,195.873688,195.873688,2,2
4,4,platinum,blueorange,trainstation_dawn_ptrainstation_dawn_p,732,OctaneOctane,260.79,330.97,16,16,...,120.835720,79.164277,151.986890,48.013109,196.647236,196.647236,196.647236,196.647236,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30116,30116,platinum,blueorange,neotokyo_standard_pneotokyo_standard_p,656,OctaneOctane,206.86,284.36,13,13,...,120.551075,79.448913,144.377590,55.622410,193.610440,193.610440,193.610440,193.610440,0,0
30117,30117,bronze,blueorange,neotokyo_standard_pneotokyo_standard_p,852,OctaneOctane,211.08,365.22,20,20,...,128.013235,71.986763,155.357760,44.642239,195.385700,195.385700,195.385700,195.385700,1,1
30118,30118,silver,blueorange,chn_stadium_pchn_stadium_p,780,Mudcat GXTRonin GXT,249.92,344.39,14,14,...,119.381703,80.618304,142.307998,57.692005,196.069200,196.069200,196.069200,196.069200,1,1
30119,30119,silver,blueorange,trainstation_ptrainstation_p,744,FennecOctane,264.72,336.73,11,11,...,128.526040,71.473962,148.898055,51.101952,198.715385,198.715385,198.715385,198.715385,2,2


In [4]:
# Set features and target
X = match_df.drop(columns=['match_id', 'rank', 'color', 'car_name'])
y = match_df['rank']

In [5]:
le = LabelEncoder()
y = le.fit_transform(y)
le.classes_

array(['bronze', 'champion', 'diamond', 'gold', 'platinum', 'silver'],
      dtype=object)

In [6]:
# Split train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [7]:
# Initialize column transformer
ct = ColumnTransformer(
    transformers=[
        ('ohe', OneHotEncoder(), ['map_code']),
        ('imputer', SimpleImputer(), ['possession_time', 'time_in_side', 'goals_against_while_last_defender'])
    ],
    remainder='passthrough'
)

In [8]:
# Function to create the Keras model for SciKeras
def create_model():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(108,)))
    model.add(tf.keras.layers.Dense(2, activation='relu'))
    model.add(tf.keras.layers.Dense(6, activation='softmax')) # Need node per rank
    model.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Keras model with SciKeras wrapper
model = KerasClassifier(model=create_model, epochs=100)

In [9]:
# Create and fit model pipeline
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('scaler', StandardScaler()),
        ('model', model)
    ]
).fit(X_train, y_train)

Epoch 1/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.2991 - loss: 1.5450
Epoch 2/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4684 - loss: 1.2509
Epoch 3/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5047 - loss: 1.1523
Epoch 4/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5138 - loss: 1.1102
Epoch 5/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5185 - loss: 1.0845
Epoch 6/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5409 - loss: 1.0655
Epoch 7/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5475 - loss: 1.0512
Epoch 8/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5507 - loss: 1.0411
Epoch 9/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5487 - loss: 1.0337
Epoch 10/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5532 - loss: 1.0277
Epoch 11/100
659/659 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5546 - loss: 1.0226
Epoch 12/100
659/659 ━━━━━━━━━━━━━━━━━━━━

In [10]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

659/659 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [11]:
confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
)

array([[   2,    2,    3,   49,   10,  450],
       [   1, 3081,  896,   12,  119,    6],
       [   1, 1172, 2300,  122, 1225,   21],
       [   0,    5,  130, 2658, 1083,  500],
       [   1,  140, 1054, 1202, 2769,   82],
       [   5,    3,    5,  797,   58, 1120]])

In [13]:
confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)

array([[   1,    0,    1,   27,    8,  184],
       [   0, 1282,  419,    4,   57,    1],
       [   0,  546,  916,   47,  559,    7],
       [   1,    3,   59, 1129,  482,  202],
       [   0,   68,  435,  518, 1200,   29],
       [   0,    2,    3,  360,   26,  461]])

In [14]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))
print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

              precision    recall  f1-score   support

           0       0.20      0.00      0.01       516
           1       0.70      0.75      0.72      4115
           2       0.52      0.48      0.50      4841
           3       0.55      0.61      0.58      4376
           4       0.53      0.53      0.53      5248
           5       0.51      0.56      0.54      1988

    accuracy                           0.57     21084
   macro avg       0.50      0.49      0.48     21084
weighted avg       0.56      0.57      0.56     21084

              precision    recall  f1-score   support

           0       0.50      0.00      0.01       221
           1       0.67      0.73      0.70      1763
           2       0.50      0.44      0.47      2075
           3       0.54      0.60      0.57      1876
           4       0.51      0.53      0.52      2250
           5       0.52      0.54      0.53       852

    accuracy                           0.55      9037
   macro avg       0.54

In [16]:
test_df = pd.read_csv('test.csv')

In [17]:
match_test_df = test_df.groupby(['match_id']).sum().reset_index()

In [18]:
y_pred = pipe.predict(match_test_df.drop(columns=['match_id']))

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [21]:
converter = { 0: 1, 5: 2, 3: 3, 4: 4, 2: 5, 1: 6 }

y_pred = pd.Series(y_pred).map(converter)

In [23]:
submission = pd.concat([match_test_df['match_id'], y_pred], axis = 1).rename(columns = {0: 'rank'})

In [25]:
submission.to_csv('keras_all_submission.csv', index=False)

In [26]:
submission.shape

(2500, 2)